## A08+A09 - Bootstrapping & RF

In [40]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LinearRegression
from scipy import stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, r2_score
from skopt import BayesSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

In [3]:
df = pd.read_excel(r"C:\Users\valer\Motor Trend Car Road Tests.xlsx")
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


### Regresión lineal

In [5]:
X = df[['hp', 'qsec']]
y = df['mpg']

In [6]:
model = LinearRegression()
model.fit(X, y)

LinearRegression()

In [7]:
print("b0:", model.intercept_)
print("b1 (hp):", model.coef_[0])
print("b2 (qsec):", model.coef_[1])

b0: 48.32370516913445
b1 (hp): -0.08459304367409272
b2 (qsec): -0.8865796246342723


In [8]:
X_ols = sm.add_constant(X)
ols_model = sm.OLS(y, X_ols).fit()

In [9]:
print(ols_model.summary())
print("\nIntervalos de confianza:")
print(ols_model.conf_int())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        16:20:30   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

### Bootstrap

In [10]:
B = 5000
n = len(df)
betas = []
for i in range (B):
    sample = df.sample(n=n, replace=True)
    x = sample[['hp', 'qsec']]
    y = sample['mpg']
    model_b = LinearRegression()
    model_b.fit(x, y)
    b0 = model_b.intercept_
    b1, b2 = model_b.coef_
    betas.append([b0, b1, b2])
betas = np.array(betas)

In [11]:
mean = betas.mean(axis=0)
std = betas.std(axis=0)

In [12]:
print("Promedios:", mean)
print("Desviaciones:", std)

Promedios: [50.28188861 -0.08793462 -0.97411081]
Desviaciones: [11.05458355  0.0169681   0.527954  ]


In [13]:
lower = mean - 2 * std
upper = mean + 2 * std

for i, name in enumerate(['b0', 'b1 (hp)', 'b2 (qsec)']):
    print(f"{name}: [{lower[i]:.4f}, {upper[i]:.4f}]")

b0: [28.1727, 72.3911]
b1 (hp): [-0.1219, -0.0540]
b2 (qsec): [-2.0300, 0.0818]


In [14]:
print(ols_model.conf_int())

               0          1
const  25.614894  71.032516
hp     -0.113089  -0.056097
qsec   -1.979929   0.206770


### Interpretación

Al comparar los intervalos de confianza obtenidos con OLS y con bootstrap con 5000 iteraciones, se obtienen resultados muy parecidos, aunque sí presentan diferencias. Para el intercepto, ambos intervalos son cercanos, aunque el de bootstrap es un poco más amplio. En el caso de hp, los dos métodos dan intervalos negativos que no incluyen el cero, lo que indica que sí es una variable significativa y que su efecto sobre mpg es negativo. Por otra parte en el caso de qsec, en ambos métodos el intervalo incluye el cero, por lo que no resulta significativo para el modelo.

### Actividad

In [9]:
X = df.drop(columns=['mpg', 'model']) 
y = df['mpg']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [20]:
columnas = X.columns
resultados = []
B = 1000
for i in range(B):
    vars_random = np.random.choice(columnas, size=3, replace=False)
    Xi_train = X_train[list(vars_random)]
    modelo = LinearRegression()
    modelo.fit(Xi_train, y_train)
    resultados.append({
        'variables': vars_random,
        'modelo': modelo
    })

In [21]:
predicciones = []

for res in resultados:
    
    vars_modelo = res['variables']
    modelo = res['modelo']
    
    Xi_test = X_test[list(vars_modelo)]
    
    y_pred = modelo.predict(Xi_test)
    
    predicciones.append(y_pred)

predicciones = np.array(predicciones)

predicciones.shape

(1000, 16)

In [22]:
y_pred_promedio = predicciones.mean(axis=0)
y_pred_promedio

array([20.33453282, 10.35558717, 14.45315785, 27.15414079, 23.61982079,
       20.20054031, 13.43202187, 27.46346373, 15.29626395, 21.78023422,
       15.41774565, 10.53911715, 19.87433873, 15.26637089, 14.80368326,
       13.35254221])

In [23]:
r2_promedio = r2_score(y_test, y_pred_promedio)
r2_promedio

0.7850100874564069

### Random forest

In [4]:
X = df.drop(columns=['mpg', 'model']) 
y = df['mpg']

In [5]:
rf = RandomForestRegressor(random_state=42)

In [11]:
param_space = {
    'n_estimators': (5, 30),
    'max_depth': (2, 10),
    'max_leaf_nodes': (1, 20)
}

In [12]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

In [13]:
opt = BayesSearchCV(
    estimator=rf,
    search_spaces=param_space,
    n_iter=10,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

In [14]:
opt.fit(X, y)

BayesSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
              estimator=RandomForestRegressor(random_state=42), n_iter=10,
              n_jobs=-1, random_state=42, scoring='r2',
              search_spaces={'max_depth': (2, 10), 'max_leaf_nodes': (1, 20),
                             'n_estimators': (5, 30)})

In [15]:
print("Mejores hiperparámetros: ")
print(opt.best_params_)

print("\nMejor R2:")
print(opt.best_score_)

Mejores hiperparámetros: 
OrderedDict({'max_depth': 2, 'max_leaf_nodes': 17, 'n_estimators': 24})

Mejor R2:
0.49630352859834226


**Aplicación del modelo limpio en el dataset**

In [16]:
rf2 = RandomForestRegressor(max_depth=2, max_leaf_nodes=17, n_estimators=24)

In [17]:
rf2.fit(X, y)

RandomForestRegressor(max_depth=2, max_leaf_nodes=17, n_estimators=24)

In [18]:
y_pred = rf2.predict(X)
y_pred

array([21.31105249, 21.31105249, 22.58584085, 19.92552403, 16.60051453,
       19.25369503, 14.96114208, 22.32256043, 21.73885011, 18.59465089,
       18.59465089, 16.16849072, 16.16849072, 16.16849072, 14.30616628,
       14.18856212, 14.46633989, 28.94967923, 29.95826389, 30.36868056,
       21.78579456, 16.60051453, 16.60051453, 14.96114208, 16.60051453,
       28.6719213 , 25.94858343, 27.48937017, 16.18970329, 20.15228613,
       15.17054684, 21.96426678])

In [31]:
r2 = r2_score(y, y_pred)
r2

0.9153068662826419

### Gradient Boosting Regressor

In [32]:
X = df.drop(columns=['mpg', 'model']) 
y = df['mpg']

In [37]:
gbr = GradientBoostingRegressor(random_state=42)

In [47]:
param_space = {
    'n_estimators': (5, 30),
    'max_depth': (2, 10),
    'max_leaf_nodes': (1, 20)
}

In [48]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

In [49]:
opt = BayesSearchCV(
    estimator=rf,
    search_spaces=param_space,
    n_iter=10,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

In [50]:
opt.fit(X, y)

BayesSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
              estimator=RandomForestRegressor(random_state=42), n_iter=10,
              n_jobs=-1, random_state=42, scoring='r2',
              search_spaces={'max_depth': (2, 10), 'max_leaf_nodes': (1, 20),
                             'n_estimators': (5, 30)})

In [52]:
print("Mejores hiperparámetros:")
print(grid.best_params_)

print("\nMejor R2 (CV):")
print(grid.best_score_)

Mejores hiperparámetros:
{'max_depth': 5, 'max_leaf_nodes': None, 'n_estimators': 100}

Mejor R2 (CV):
0.27030285469736326


**Aplicación del modelo limpio en el dataset**

In [60]:
gbr2 = RandomForestRegressor(max_depth=5, max_leaf_nodes=None, n_estimators=100)

In [61]:
gbr2.fit(X, y)

RandomForestRegressor(max_depth=5)

In [62]:
y_pred = gbr2.predict(X)
y_pred

array([20.91672063, 20.88947063, 23.97033333, 20.06057738, 18.07760238,
       18.75505476, 14.79490714, 23.52978968, 22.62242857, 18.83545541,
       18.36408874, 16.16140296, 16.29348694, 15.72563694, 11.4185    ,
       11.068     , 13.89987381, 30.943     , 30.759     , 32.804     ,
       21.92534563, 16.11305123, 16.3169298 , 14.11023391, 17.94087619,
       29.103     , 25.39775   , 28.34616667, 16.0503917 , 19.61599192,
       14.8533417 , 21.6782623 ])

In [64]:
r2 = r2_score(y, y_pred)
r2

0.9782835230427704

________________________________
## Conclusión

En este trabajo se aplicaron distintos métodos de remuestreo y modelos de aprendizaje estadístico con el objetivo de evaluar y mejorar la capacidad predictiva de los modelos. A lo largo de los ejercicios se utilizaron técnicas como validation set, bootstrap y cross-validation, lo que permitió estimar errores, medir la variabilidad y validar el desempeño de los modelos de manera.

Asimismo, se implementaron métodos de ensemble como aggregating, Random Forest y Gradient Boosting, los cuales combinan múltiples modelos para reducir la variabilidad y mejorar la precisión de las predicciones. En particular, se observó que el uso de múltiples modelos y la optimización de hiperparámetros mediante técnicas como GridSearch o búsqueda bayesiana permiten obtener mejores resultados en términos de R^2.

En general, los resultados muestran la importancia de validar adecuadamente los modelos y de utilizar métodos más avanzados cuando se busca mejorar su desempeño. Estos enfoques no solo incrementan la precisión, sino que también permiten obtener modelos más estables y confiables.